In [ ]:
import os
import numpy as np
import pandas as pd
import pickle

from code_libraries.training_metrics import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.calibration import calibration_curve

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input
from tensorflow.keras.optimizers import Adam

In [ ]:
main_dir = "../"
data_arrays = os.path.join(main_dir , "heavy_data_NO_GIT/data_arrays")

In [ ]:
random_state = 76
test_size = 0.20

sampling_fraction = 1.0
epochs = 5

## RAW

In [ ]:
df_path_sn1_raw = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_raw.csv")
df_path_sn2_raw = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_raw.csv")

df_sn1_raw = pd.read_csv(df_path_sn1_raw)
df_sn2_raw = pd.read_csv(df_path_sn2_raw)

df_sn1_raw = df_sn1_raw.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_raw = df_sn2_raw.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_raw = pd.read_csv(df_path_sn1_raw)
df_sn2_raw = pd.read_csv(df_path_sn2_raw)

df_sn1_raw.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_raw.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_raw = df_sn1_raw.iloc[: , :-1]
y_sn1_raw= df_sn1_raw.iloc[: , -1]

X_sn1_raw_train, X_sn1_raw_test, y_sn1_raw_train , y_sn1_raw_test = train_test_split(X_sn1_raw , y_sn1_raw , test_size=test_size, shuffle=True)

scaler_sn1_raw = StandardScaler()

X_sn1_raw_train_scaled = scaler_sn1_raw.fit_transform(X_sn1_raw_train)
X_sn1_raw_test_scaled = scaler_sn1_raw.transform(X_sn1_raw_test)

nn_sn1_raw = Sequential([
    Input(shape=(X_sn1_raw_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_raw.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_raw.fit(
    X_sn1_raw_train_scaled,
    y_sn1_raw_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_raw_pred_prob = nn_sn1_raw.predict(X_sn1_raw_test_scaled).flatten()
nn_sn1_raw_pred = (nn_sn1_raw_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_raw_pred_prob


nn_acc_sn1_raw = accuracy_score(y_sn1_raw_test, nn_sn1_raw_pred)
print("Test accuracy:", nn_acc_sn1_raw)


In [ ]:
X_sn2_raw = df_sn2_raw.iloc[: , :-1]
y_sn2_raw= df_sn2_raw.iloc[: , -1]

X_sn2_raw_train, X_sn2_raw_test, y_sn2_raw_train , y_sn2_raw_test = train_test_split(X_sn2_raw , y_sn2_raw , test_size=test_size, shuffle=True)

scaler_sn2_raw = StandardScaler()

X_sn2_raw_train_scaled = scaler_sn2_raw.fit_transform(X_sn2_raw_train)
X_sn2_raw_test_scaled = scaler_sn2_raw.transform(X_sn2_raw_test)

nn_sn2_raw = Sequential([
    Input(shape=(X_sn2_raw_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_raw.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_raw.fit(
    X_sn2_raw_train_scaled,
    y_sn2_raw_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_raw_pred_prob = nn_sn2_raw.predict(X_sn2_raw_test_scaled).flatten()
nn_sn2_raw_pred = (nn_sn2_raw_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_raw_pred_prob


nn_acc_sn2_raw = accuracy_score(y_sn2_raw_test, nn_sn2_raw_pred)
print("Test accuracy:", nn_acc_sn2_raw)


In [ ]:
test_sn1_raw = (X_sn1_raw_test , y_sn1_raw_test)

with open(os.path.join(data_arrays , "nn_test_sn1_raw.pkl") , "wb") as f:
    pickle.dump(test_sn1_raw, f)

test_sn2_raw = (X_sn2_raw_test , y_sn2_raw_test)

with open(os.path.join(data_arrays , "nn_test_sn2_raw.pkl") , "wb") as f:
    pickle.dump(test_sn2_raw, f)

filename_nn_model_sn1_raw = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_raw_model_ft_last.sav')
filename_scaler_sn1_raw = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_raw_scaler_ft_last.sav')
pickle.dump(nn_sn1_raw, open(filename_nn_model_sn1_raw, 'wb'))
pickle.dump(scaler_sn1_raw, open(filename_scaler_sn1_raw, 'wb'))

filename_nn_model_sn2_raw = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_raw_model_ft_last.sav')
filename_scaler_sn2_raw = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_raw_scaler_ft_last.sav')
pickle.dump(nn_sn2_raw, open(filename_nn_model_sn2_raw, 'wb'))
pickle.dump(scaler_sn2_raw, open(filename_scaler_sn2_raw, 'wb'))


## 3pt

In [ ]:
df_path_sn1_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_3pt.csv")
df_path_sn2_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_3pt.csv")

df_sn1_3pt = pd.read_csv(df_path_sn1_3pt)
df_sn2_3pt = pd.read_csv(df_path_sn2_3pt)

df_sn1_3pt = df_sn1_3pt.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_3pt = df_sn2_3pt.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_3pt = pd.read_csv(df_path_sn1_3pt)
df_sn2_3pt = pd.read_csv(df_path_sn2_3pt)

df_sn1_3pt.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_3pt.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_3pt = df_sn1_3pt.iloc[: , :-1]
y_sn1_3pt= df_sn1_3pt.iloc[: , -1]

X_sn1_3pt_train, X_sn1_3pt_test, y_sn1_3pt_train , y_sn1_3pt_test = train_test_split(X_sn1_3pt , y_sn1_3pt , test_size=test_size, shuffle=True)

scaler_sn1_3pt = StandardScaler()

X_sn1_3pt_train_scaled = scaler_sn1_3pt.fit_transform(X_sn1_3pt_train)
X_sn1_3pt_test_scaled = scaler_sn1_3pt.transform(X_sn1_3pt_test)

nn_sn1_3pt = Sequential([
    Input(shape=(X_sn1_3pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_3pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_3pt.fit(
    X_sn1_3pt_train_scaled,
    y_sn1_3pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_3pt_pred_prob = nn_sn1_3pt.predict(X_sn1_3pt_test_scaled).flatten()
nn_sn1_3pt_pred = (nn_sn1_3pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_3pt_pred_prob


nn_acc_sn1_3pt = accuracy_score(y_sn1_3pt_test, nn_sn1_3pt_pred)
print("Test accuracy:", nn_acc_sn1_3pt)


In [ ]:
X_sn2_3pt = df_sn2_3pt.iloc[: , :-1]
y_sn2_3pt= df_sn2_3pt.iloc[: , -1]

X_sn2_3pt_train, X_sn2_3pt_test, y_sn2_3pt_train , y_sn2_3pt_test = train_test_split(X_sn2_3pt , y_sn2_3pt , test_size=test_size, shuffle=True)

scaler_sn2_3pt = StandardScaler()

X_sn2_3pt_train_scaled = scaler_sn2_3pt.fit_transform(X_sn2_3pt_train)
X_sn2_3pt_test_scaled = scaler_sn2_3pt.transform(X_sn2_3pt_test)

nn_sn2_3pt = Sequential([
    Input(shape=(X_sn2_3pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_3pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_3pt.fit(
    X_sn2_3pt_train_scaled,
    y_sn2_3pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_3pt_pred_prob = nn_sn2_3pt.predict(X_sn2_3pt_test_scaled).flatten()
nn_sn2_3pt_pred = (nn_sn2_3pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_3pt_pred_prob


nn_acc_sn2_3pt = accuracy_score(y_sn2_3pt_test, nn_sn2_3pt_pred)
print("Test accuracy:", nn_acc_sn2_3pt)


In [ ]:
test_sn1_3pt = (X_sn1_3pt_test , y_sn1_3pt_test)

with open(os.path.join(data_arrays , "nn_test_sn1_3pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_3pt, f)

test_sn2_3pt = (X_sn2_3pt_test , y_sn2_3pt_test)

with open(os.path.join(data_arrays , "nn_test_sn2_3pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_3pt, f)

filename_nn_model_sn1_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_3pt_model_ft_last.sav')
filename_scaler_sn1_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_3pt_scaler_ft_last.sav')
pickle.dump(nn_sn1_3pt, open(filename_nn_model_sn1_3pt, 'wb'))
pickle.dump(scaler_sn1_3pt, open(filename_scaler_sn1_3pt, 'wb'))

filename_nn_model_sn2_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_3pt_model_ft_last.sav')
filename_scaler_sn2_3pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_3pt_scaler_ft_last.sav')
pickle.dump(nn_sn2_3pt, open(filename_nn_model_sn2_3pt, 'wb'))
pickle.dump(scaler_sn2_3pt, open(filename_scaler_sn2_3pt, 'wb'))


## 5pt

In [ ]:
df_path_sn1_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_5pt.csv")
df_path_sn2_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_5pt.csv")

df_sn1_5pt = pd.read_csv(df_path_sn1_5pt)
df_sn2_5pt = pd.read_csv(df_path_sn2_5pt)

df_sn1_5pt = df_sn1_5pt.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_5pt = df_sn2_5pt.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_5pt = pd.read_csv(df_path_sn1_5pt)
df_sn2_5pt = pd.read_csv(df_path_sn2_5pt)

df_sn1_5pt.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_5pt.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_5pt = df_sn1_5pt.iloc[: , :-1]
y_sn1_5pt= df_sn1_5pt.iloc[: , -1]

X_sn1_5pt_train, X_sn1_5pt_test, y_sn1_5pt_train , y_sn1_5pt_test = train_test_split(X_sn1_5pt , y_sn1_5pt , test_size=test_size, shuffle=True)

scaler_sn1_5pt = StandardScaler()

X_sn1_5pt_train_scaled = scaler_sn1_5pt.fit_transform(X_sn1_5pt_train)
X_sn1_5pt_test_scaled = scaler_sn1_5pt.transform(X_sn1_5pt_test)

nn_sn1_5pt = Sequential([
    Input(shape=(X_sn1_5pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_5pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_5pt.fit(
    X_sn1_5pt_train_scaled,
    y_sn1_5pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_5pt_pred_prob = nn_sn1_5pt.predict(X_sn1_5pt_test_scaled).flatten()
nn_sn1_5pt_pred = (nn_sn1_5pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_5pt_pred_prob


nn_acc_sn1_5pt = accuracy_score(y_sn1_5pt_test, nn_sn1_5pt_pred)
print("Test accuracy:", nn_acc_sn1_5pt)


In [ ]:
X_sn2_5pt = df_sn2_5pt.iloc[: , :-1]
y_sn2_5pt= df_sn2_5pt.iloc[: , -1]

X_sn2_5pt_train, X_sn2_5pt_test, y_sn2_5pt_train , y_sn2_5pt_test = train_test_split(X_sn2_5pt , y_sn2_5pt , test_size=test_size, shuffle=True)

scaler_sn2_5pt = StandardScaler()

X_sn2_5pt_train_scaled = scaler_sn2_5pt.fit_transform(X_sn2_5pt_train)
X_sn2_5pt_test_scaled = scaler_sn2_5pt.transform(X_sn2_5pt_test)

nn_sn2_5pt = Sequential([
    Input(shape=(X_sn2_5pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_5pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_5pt.fit(
    X_sn2_5pt_train_scaled,
    y_sn2_5pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_5pt_pred_prob = nn_sn2_5pt.predict(X_sn2_5pt_test_scaled).flatten()
nn_sn2_5pt_pred = (nn_sn2_5pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_5pt_pred_prob


nn_acc_sn2_5pt = accuracy_score(y_sn2_5pt_test, nn_sn2_5pt_pred)
print("Test accuracy:", nn_acc_sn2_5pt)


In [ ]:
test_sn1_5pt = (X_sn1_5pt_test , y_sn1_5pt_test)

with open(os.path.join(data_arrays , "nn_test_sn1_5pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_5pt, f)

test_sn2_5pt = (X_sn2_5pt_test , y_sn2_5pt_test)

with open(os.path.join(data_arrays , "nn_test_sn2_5pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_5pt, f)

filename_nn_model_sn1_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_5pt_model_ft_last.sav')
filename_scaler_sn1_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_5pt_scaler_ft_last.sav')
pickle.dump(nn_sn1_5pt, open(filename_nn_model_sn1_5pt, 'wb'))
pickle.dump(scaler_sn1_5pt, open(filename_scaler_sn1_5pt, 'wb'))

filename_nn_model_sn2_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_5pt_model_ft_last.sav')
filename_scaler_sn2_5pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_5pt_scaler_ft_last.sav')
pickle.dump(nn_sn2_5pt, open(filename_nn_model_sn2_5pt, 'wb'))
pickle.dump(scaler_sn2_5pt, open(filename_scaler_sn2_5pt, 'wb'))


## 7pt

In [ ]:
df_path_sn1_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_7pt.csv")
df_path_sn2_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_7pt.csv")

df_sn1_7pt = pd.read_csv(df_path_sn1_7pt)
df_sn2_7pt = pd.read_csv(df_path_sn2_7pt)

df_sn1_7pt = df_sn1_7pt.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_7pt = df_sn2_7pt.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_7pt = pd.read_csv(df_path_sn1_7pt)
df_sn2_7pt = pd.read_csv(df_path_sn2_7pt)

df_sn1_7pt.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_7pt.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_7pt = df_sn1_7pt.iloc[: , :-1]
y_sn1_7pt= df_sn1_7pt.iloc[: , -1]

X_sn1_7pt_train, X_sn1_7pt_test, y_sn1_7pt_train , y_sn1_7pt_test = train_test_split(X_sn1_7pt , y_sn1_7pt , test_size=test_size, shuffle=True)

scaler_sn1_7pt = StandardScaler()

X_sn1_7pt_train_scaled = scaler_sn1_7pt.fit_transform(X_sn1_7pt_train)
X_sn1_7pt_test_scaled = scaler_sn1_7pt.transform(X_sn1_7pt_test)

nn_sn1_7pt = Sequential([
    Input(shape=(X_sn1_7pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_7pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_7pt.fit(
    X_sn1_7pt_train_scaled,
    y_sn1_7pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_7pt_pred_prob = nn_sn1_7pt.predict(X_sn1_7pt_test_scaled).flatten()
nn_sn1_7pt_pred = (nn_sn1_7pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_7pt_pred_prob


nn_acc_sn1_7pt = accuracy_score(y_sn1_7pt_test, nn_sn1_7pt_pred)
print("Test accuracy:", nn_acc_sn1_7pt)


In [ ]:
X_sn2_7pt = df_sn2_7pt.iloc[: , :-1]
y_sn2_7pt= df_sn2_7pt.iloc[: , -1]

X_sn2_7pt_train, X_sn2_7pt_test, y_sn2_7pt_train , y_sn2_7pt_test = train_test_split(X_sn2_7pt , y_sn2_7pt , test_size=test_size, shuffle=True)

scaler_sn2_7pt = StandardScaler()

X_sn2_7pt_train_scaled = scaler_sn2_7pt.fit_transform(X_sn2_7pt_train)
X_sn2_7pt_test_scaled = scaler_sn2_7pt.transform(X_sn2_7pt_test)

nn_sn2_7pt = Sequential([
    Input(shape=(X_sn2_7pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_7pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_7pt.fit(
    X_sn2_7pt_train_scaled,
    y_sn2_7pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_7pt_pred_prob = nn_sn2_7pt.predict(X_sn2_7pt_test_scaled).flatten()
nn_sn2_7pt_pred = (nn_sn2_7pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_7pt_pred_prob


nn_acc_sn2_7pt = accuracy_score(y_sn2_7pt_test, nn_sn2_7pt_pred)
print("Test accuracy:", nn_acc_sn2_7pt)


In [ ]:
test_sn1_7pt = (X_sn1_7pt_test , y_sn1_7pt_test)

with open(os.path.join(data_arrays , "nn_test_sn1_7pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_7pt, f)

test_sn2_7pt = (X_sn2_7pt_test , y_sn2_7pt_test)

with open(os.path.join(data_arrays , "nn_test_sn2_7pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_7pt, f)

filename_nn_model_sn1_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_7pt_model_ft_last.sav')
filename_scaler_sn1_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_7pt_scaler_ft_last.sav')
pickle.dump(nn_sn1_7pt, open(filename_nn_model_sn1_7pt, 'wb'))
pickle.dump(scaler_sn1_7pt, open(filename_scaler_sn1_7pt, 'wb'))

filename_nn_model_sn2_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_7pt_model_ft_last.sav')
filename_scaler_sn2_7pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_7pt_scaler_ft_last.sav')
pickle.dump(nn_sn2_7pt, open(filename_nn_model_sn2_7pt, 'wb'))
pickle.dump(scaler_sn2_7pt, open(filename_scaler_sn2_7pt, 'wb'))


## 9pt

In [ ]:
df_path_sn1_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_9pt.csv")
df_path_sn2_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_9pt.csv")

df_sn1_9pt = pd.read_csv(df_path_sn1_9pt)
df_sn2_9pt = pd.read_csv(df_path_sn2_9pt)

df_sn1_9pt = df_sn1_9pt.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_9pt = df_sn2_9pt.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_9pt = pd.read_csv(df_path_sn1_9pt)
df_sn2_9pt = pd.read_csv(df_path_sn2_9pt)

df_sn1_9pt.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_9pt.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_9pt = df_sn1_9pt.iloc[: , :-1]
y_sn1_9pt= df_sn1_9pt.iloc[: , -1]

X_sn1_9pt_train, X_sn1_9pt_test, y_sn1_9pt_train , y_sn1_9pt_test = train_test_split(X_sn1_9pt , y_sn1_9pt , test_size=test_size, shuffle=True)

scaler_sn1_9pt = StandardScaler()

X_sn1_9pt_train_scaled = scaler_sn1_9pt.fit_transform(X_sn1_9pt_train)
X_sn1_9pt_test_scaled = scaler_sn1_9pt.transform(X_sn1_9pt_test)

nn_sn1_9pt = Sequential([
    Input(shape=(X_sn1_9pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_9pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_9pt.fit(
    X_sn1_9pt_train_scaled,
    y_sn1_9pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_9pt_pred_prob = nn_sn1_9pt.predict(X_sn1_9pt_test_scaled).flatten()
nn_sn1_9pt_pred = (nn_sn1_9pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_9pt_pred_prob


nn_acc_sn1_9pt = accuracy_score(y_sn1_9pt_test, nn_sn1_9pt_pred)
print("Test accuracy:", nn_acc_sn1_9pt)


In [ ]:
X_sn2_9pt = df_sn2_9pt.iloc[: , :-1]
y_sn2_9pt= df_sn2_9pt.iloc[: , -1]

X_sn2_9pt_train, X_sn2_9pt_test, y_sn2_9pt_train , y_sn2_9pt_test = train_test_split(X_sn2_9pt , y_sn2_9pt , test_size=test_size, shuffle=True)

scaler_sn2_9pt = StandardScaler()

X_sn2_9pt_train_scaled = scaler_sn2_9pt.fit_transform(X_sn2_9pt_train)
X_sn2_9pt_test_scaled = scaler_sn2_9pt.transform(X_sn2_9pt_test)

nn_sn2_9pt = Sequential([
    Input(shape=(X_sn2_9pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_9pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_9pt.fit(
    X_sn2_9pt_train_scaled,
    y_sn2_9pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_9pt_pred_prob = nn_sn2_9pt.predict(X_sn2_9pt_test_scaled).flatten()
nn_sn2_9pt_pred = (nn_sn2_9pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_9pt_pred_prob


nn_acc_sn2_9pt = accuracy_score(y_sn2_9pt_test, nn_sn2_9pt_pred)
print("Test accuracy:", nn_acc_sn2_9pt)


In [ ]:
test_sn1_9pt = (X_sn1_9pt_test , y_sn1_9pt_test)

with open(os.path.join(data_arrays , "nn_test_sn1_9pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_9pt, f)

test_sn2_9pt = (X_sn2_9pt_test , y_sn2_9pt_test)

with open(os.path.join(data_arrays , "nn_test_sn2_9pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_9pt, f)

filename_nn_model_sn1_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_9pt_model_ft_last.sav')
filename_scaler_sn1_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_9pt_scaler_ft_last.sav')
pickle.dump(nn_sn1_9pt, open(filename_nn_model_sn1_9pt, 'wb'))
pickle.dump(scaler_sn1_9pt, open(filename_scaler_sn1_9pt, 'wb'))

filename_nn_model_sn2_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_9pt_model_ft_last.sav')
filename_scaler_sn2_9pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_9pt_scaler_ft_last.sav')
pickle.dump(nn_sn2_9pt, open(filename_nn_model_sn2_9pt, 'wb'))
pickle.dump(scaler_sn2_9pt, open(filename_scaler_sn2_9pt, 'wb'))


## 11pt

In [ ]:
df_path_sn1_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_11pt.csv")
df_path_sn2_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_11pt.csv")

df_sn1_11pt = pd.read_csv(df_path_sn1_11pt)
df_sn2_11pt = pd.read_csv(df_path_sn2_11pt)

df_sn1_11pt = df_sn1_11pt.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_11pt = df_sn2_11pt.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_11pt = pd.read_csv(df_path_sn1_11pt)
df_sn2_11pt = pd.read_csv(df_path_sn2_11pt)

df_sn1_11pt.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_11pt.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_11pt = df_sn1_11pt.iloc[: , :-1]
y_sn1_11pt= df_sn1_11pt.iloc[: , -1]

X_sn1_11pt_train, X_sn1_11pt_test, y_sn1_11pt_train , y_sn1_11pt_test = train_test_split(X_sn1_11pt , y_sn1_11pt , test_size=test_size, shuffle=True)

scaler_sn1_11pt = StandardScaler()

X_sn1_11pt_train_scaled = scaler_sn1_11pt.fit_transform(X_sn1_11pt_train)
X_sn1_11pt_test_scaled = scaler_sn1_11pt.transform(X_sn1_11pt_test)

nn_sn1_11pt = Sequential([
    Input(shape=(X_sn1_11pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_11pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_11pt.fit(
    X_sn1_11pt_train_scaled,
    y_sn1_11pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_11pt_pred_prob = nn_sn1_11pt.predict(X_sn1_11pt_test_scaled).flatten()
nn_sn1_11pt_pred = (nn_sn1_11pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_11pt_pred_prob


nn_acc_sn1_11pt = accuracy_score(y_sn1_11pt_test, nn_sn1_11pt_pred)
print("Test accuracy:", nn_acc_sn1_11pt)


In [ ]:
X_sn2_11pt = df_sn2_11pt.iloc[: , :-1]
y_sn2_11pt= df_sn2_11pt.iloc[: , -1]

X_sn2_11pt_train, X_sn2_11pt_test, y_sn2_11pt_train , y_sn2_11pt_test = train_test_split(X_sn2_11pt , y_sn2_11pt , test_size=test_size, shuffle=True)

scaler_sn2_11pt = StandardScaler()

X_sn2_11pt_train_scaled = scaler_sn2_11pt.fit_transform(X_sn2_11pt_train)
X_sn2_11pt_test_scaled = scaler_sn2_11pt.transform(X_sn2_11pt_test)

nn_sn2_11pt = Sequential([
    Input(shape=(X_sn2_11pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_11pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_11pt.fit(
    X_sn2_11pt_train_scaled,
    y_sn2_11pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_11pt_pred_prob = nn_sn2_11pt.predict(X_sn2_11pt_test_scaled).flatten()
nn_sn2_11pt_pred = (nn_sn2_11pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_11pt_pred_prob


nn_acc_sn2_11pt = accuracy_score(y_sn2_11pt_test, nn_sn2_11pt_pred)
print("Test accuracy:", nn_acc_sn2_11pt)


In [ ]:
test_sn1_11pt = (X_sn1_11pt_test , y_sn1_11pt_test)

with open(os.path.join(data_arrays , "nn_test_sn1_11pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_11pt, f)

test_sn2_11pt = (X_sn2_11pt_test , y_sn2_11pt_test)

with open(os.path.join(data_arrays , "nn_test_sn2_11pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_11pt, f)

filename_nn_model_sn1_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_11pt_model_ft_last.sav')
filename_scaler_sn1_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_11pt_scaler_ft_last.sav')
pickle.dump(nn_sn1_11pt, open(filename_nn_model_sn1_11pt, 'wb'))
pickle.dump(scaler_sn1_11pt, open(filename_scaler_sn1_11pt, 'wb'))

filename_nn_model_sn2_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_11pt_model_ft_last.sav')
filename_scaler_sn2_11pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_11pt_scaler_ft_last.sav')
pickle.dump(nn_sn2_11pt, open(filename_nn_model_sn2_11pt, 'wb'))
pickle.dump(scaler_sn2_11pt, open(filename_scaler_sn2_11pt, 'wb'))


## 13pt

In [ ]:
df_path_sn1_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_13pt.csv")
df_path_sn2_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_13pt.csv")

df_sn1_13pt = pd.read_csv(df_path_sn1_13pt)
df_sn2_13pt = pd.read_csv(df_path_sn2_13pt)

df_sn1_13pt = df_sn1_13pt.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_13pt = df_sn2_13pt.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_13pt = pd.read_csv(df_path_sn1_13pt)
df_sn2_13pt = pd.read_csv(df_path_sn2_13pt)

df_sn1_13pt.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_13pt.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_13pt = df_sn1_13pt.iloc[: , :-1]
y_sn1_13pt= df_sn1_13pt.iloc[: , -1]

X_sn1_13pt_train, X_sn1_13pt_test, y_sn1_13pt_train , y_sn1_13pt_test = train_test_split(X_sn1_13pt , y_sn1_13pt , test_size=test_size, shuffle=True)

scaler_sn1_13pt = StandardScaler()

X_sn1_13pt_train_scaled = scaler_sn1_13pt.fit_transform(X_sn1_13pt_train)
X_sn1_13pt_test_scaled = scaler_sn1_13pt.transform(X_sn1_13pt_test)

nn_sn1_13pt = Sequential([
    Input(shape=(X_sn1_13pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_13pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_13pt.fit(
    X_sn1_13pt_train_scaled,
    y_sn1_13pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_13pt_pred_prob = nn_sn1_13pt.predict(X_sn1_13pt_test_scaled).flatten()
nn_sn1_13pt_pred = (nn_sn1_13pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_13pt_pred_prob


nn_acc_sn1_13pt = accuracy_score(y_sn1_13pt_test, nn_sn1_13pt_pred)
print("Test accuracy:", nn_acc_sn1_13pt)


In [ ]:
X_sn2_13pt = df_sn2_13pt.iloc[: , :-1]
y_sn2_13pt= df_sn2_13pt.iloc[: , -1]

X_sn2_13pt_train, X_sn2_13pt_test, y_sn2_13pt_train , y_sn2_13pt_test = train_test_split(X_sn2_13pt , y_sn2_13pt , test_size=test_size, shuffle=True)

scaler_sn2_13pt = StandardScaler()

X_sn2_13pt_train_scaled = scaler_sn2_13pt.fit_transform(X_sn2_13pt_train)
X_sn2_13pt_test_scaled = scaler_sn2_13pt.transform(X_sn2_13pt_test)

nn_sn2_13pt = Sequential([
    Input(shape=(X_sn2_13pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_13pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_13pt.fit(
    X_sn2_13pt_train_scaled,
    y_sn2_13pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_13pt_pred_prob = nn_sn2_13pt.predict(X_sn2_13pt_test_scaled).flatten()
nn_sn2_13pt_pred = (nn_sn2_13pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_13pt_pred_prob


nn_acc_sn2_13pt = accuracy_score(y_sn2_13pt_test, nn_sn2_13pt_pred)
print("Test accuracy:", nn_acc_sn2_13pt)


In [ ]:
test_sn1_13pt = (X_sn1_13pt_test , y_sn1_13pt_test)

with open(os.path.join(data_arrays , "nn_test_sn1_13pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_13pt, f)

test_sn2_13pt = (X_sn2_13pt_test , y_sn2_13pt_test)

with open(os.path.join(data_arrays , "nn_test_sn2_13pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_13pt, f)

filename_nn_model_sn1_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_13pt_model_ft_last.sav')
filename_scaler_sn1_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_13pt_scaler_ft_last.sav')
pickle.dump(nn_sn1_13pt, open(filename_nn_model_sn1_13pt, 'wb'))
pickle.dump(scaler_sn1_13pt, open(filename_scaler_sn1_13pt, 'wb'))

filename_nn_model_sn2_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_13pt_model_ft_last.sav')
filename_scaler_sn2_13pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_13pt_scaler_ft_last.sav')
pickle.dump(nn_sn2_13pt, open(filename_nn_model_sn2_13pt, 'wb'))
pickle.dump(scaler_sn2_13pt, open(filename_scaler_sn2_13pt, 'wb'))


## 15pt

In [ ]:
df_path_sn1_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_15pt.csv")
df_path_sn2_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_15pt.csv")

df_sn1_15pt = pd.read_csv(df_path_sn1_15pt)
df_sn2_15pt = pd.read_csv(df_path_sn2_15pt)

df_sn1_15pt = df_sn1_15pt.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_15pt = df_sn2_15pt.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_15pt = pd.read_csv(df_path_sn1_15pt)
df_sn2_15pt = pd.read_csv(df_path_sn2_15pt)

df_sn1_15pt.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_15pt.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_15pt = df_sn1_15pt.iloc[: , :-1]
y_sn1_15pt= df_sn1_15pt.iloc[: , -1]

X_sn1_15pt_train, X_sn1_15pt_test, y_sn1_15pt_train , y_sn1_15pt_test = train_test_split(X_sn1_15pt , y_sn1_15pt , test_size=test_size, shuffle=True)

scaler_sn1_15pt = StandardScaler()

X_sn1_15pt_train_scaled = scaler_sn1_15pt.fit_transform(X_sn1_15pt_train)
X_sn1_15pt_test_scaled = scaler_sn1_15pt.transform(X_sn1_15pt_test)

nn_sn1_15pt = Sequential([
    Input(shape=(X_sn1_15pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_15pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_15pt.fit(
    X_sn1_15pt_train_scaled,
    y_sn1_15pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_15pt_pred_prob = nn_sn1_15pt.predict(X_sn1_15pt_test_scaled).flatten()
nn_sn1_15pt_pred = (nn_sn1_15pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_15pt_pred_prob


nn_acc_sn1_15pt = accuracy_score(y_sn1_15pt_test, nn_sn1_15pt_pred)
print("Test accuracy:", nn_acc_sn1_15pt)


In [ ]:
X_sn2_15pt = df_sn2_15pt.iloc[: , :-1]
y_sn2_15pt= df_sn2_15pt.iloc[: , -1]

X_sn2_15pt_train, X_sn2_15pt_test, y_sn2_15pt_train , y_sn2_15pt_test = train_test_split(X_sn2_15pt , y_sn2_15pt , test_size=test_size, shuffle=True)

scaler_sn2_15pt = StandardScaler()

X_sn2_15pt_train_scaled = scaler_sn2_15pt.fit_transform(X_sn2_15pt_train)
X_sn2_15pt_test_scaled = scaler_sn2_15pt.transform(X_sn2_15pt_test)

nn_sn2_15pt = Sequential([
    Input(shape=(X_sn2_15pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_15pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_15pt.fit(
    X_sn2_15pt_train_scaled,
    y_sn2_15pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_15pt_pred_prob = nn_sn2_15pt.predict(X_sn2_15pt_test_scaled).flatten()
nn_sn2_15pt_pred = (nn_sn2_15pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_15pt_pred_prob


nn_acc_sn2_15pt = accuracy_score(y_sn2_15pt_test, nn_sn2_15pt_pred)
print("Test accuracy:", nn_acc_sn2_15pt)


In [ ]:
test_sn1_15pt = (X_sn1_15pt_test , y_sn1_15pt_test)

with open(os.path.join(data_arrays , "nn_test_sn1_15pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_15pt, f)

test_sn2_15pt = (X_sn2_15pt_test , y_sn2_15pt_test)

with open(os.path.join(data_arrays , "nn_test_sn2_15pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_15pt, f)

filename_nn_model_sn1_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_15pt_model_ft_last.sav')
filename_scaler_sn1_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_15pt_scaler_ft_last.sav')
pickle.dump(nn_sn1_15pt, open(filename_nn_model_sn1_15pt, 'wb'))
pickle.dump(scaler_sn1_15pt, open(filename_scaler_sn1_15pt, 'wb'))

filename_nn_model_sn2_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_15pt_model_ft_last.sav')
filename_scaler_sn2_15pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_15pt_scaler_ft_last.sav')
pickle.dump(nn_sn2_15pt, open(filename_nn_model_sn2_15pt, 'wb'))
pickle.dump(scaler_sn2_15pt, open(filename_scaler_sn2_15pt, 'wb'))


## 17pt

In [ ]:
df_path_sn1_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn1_ft_17pt.csv")
df_path_sn2_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/dataframes/sym/training_sn2_ft_17pt.csv")

df_sn1_17pt = pd.read_csv(df_path_sn1_17pt)
df_sn2_17pt = pd.read_csv(df_path_sn2_17pt)

df_sn1_17pt = df_sn1_17pt.sample(frac = sampling_fraction).reset_index(drop=True)
df_sn2_17pt = df_sn2_17pt.sample(frac = sampling_fraction).reset_index(drop=True)

df_sn1_17pt = pd.read_csv(df_path_sn1_17pt)
df_sn2_17pt = pd.read_csv(df_path_sn2_17pt)

df_sn1_17pt.drop('Unnamed: 0' , axis = 1 , inplace=True)
df_sn2_17pt.drop('Unnamed: 0' , axis = 1 , inplace=True)


In [ ]:
X_sn1_17pt = df_sn1_17pt.iloc[: , :-1]
y_sn1_17pt= df_sn1_17pt.iloc[: , -1]

X_sn1_17pt_train, X_sn1_17pt_test, y_sn1_17pt_train , y_sn1_17pt_test = train_test_split(X_sn1_17pt , y_sn1_17pt , test_size=test_size, shuffle=True)

scaler_sn1_17pt = StandardScaler()

X_sn1_17pt_train_scaled = scaler_sn1_17pt.fit_transform(X_sn1_17pt_train)
X_sn1_17pt_test_scaled = scaler_sn1_17pt.transform(X_sn1_17pt_test)

nn_sn1_17pt = Sequential([
    Input(shape=(X_sn1_17pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn1_17pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn1_17pt.fit(
    X_sn1_17pt_train_scaled,
    y_sn1_17pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn1_17pt_pred_prob = nn_sn1_17pt.predict(X_sn1_17pt_test_scaled).flatten()
nn_sn1_17pt_pred = (nn_sn1_17pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn1_17pt_pred_prob


nn_acc_sn1_17pt = accuracy_score(y_sn1_17pt_test, nn_sn1_17pt_pred)
print("Test accuracy:", nn_acc_sn1_17pt)


In [ ]:
X_sn2_17pt = df_sn2_17pt.iloc[: , :-1]
y_sn2_17pt= df_sn2_17pt.iloc[: , -1]

X_sn2_17pt_train, X_sn2_17pt_test, y_sn2_17pt_train , y_sn2_17pt_test = train_test_split(X_sn2_17pt , y_sn2_17pt , test_size=test_size, shuffle=True)

scaler_sn2_17pt = StandardScaler()

X_sn2_17pt_train_scaled = scaler_sn2_17pt.fit_transform(X_sn2_17pt_train)
X_sn2_17pt_test_scaled = scaler_sn2_17pt.transform(X_sn2_17pt_test)

nn_sn2_17pt = Sequential([
    Input(shape=(X_sn2_17pt_train_scaled.shape[1],)),  # input layer
    Dense(34, activation='relu'),
    Dropout(0.2),
    Dense(68, activation='relu'),
    Dropout(0.2),
    Dense(34, activation='relu'),
    Dropout(0.2),
    # Dense(17, activation='relu'),
    # Dropout(0.1),
    Dense(1, activation='sigmoid')  # output
])

nn_sn2_17pt.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn_sn2_17pt.fit(
    X_sn2_17pt_train_scaled,
    y_sn2_17pt_train,
    validation_split = 0.2,
    epochs=epochs,
    batch_size=8,
    verbose=1
)

nn_sn2_17pt_pred_prob = nn_sn2_17pt.predict(X_sn2_17pt_test_scaled).flatten()
nn_sn2_17pt_pred = (nn_sn2_17pt_pred_prob > 0.5).astype(int)  # <-- use nn_sn2_17pt_pred_prob


nn_acc_sn2_17pt = accuracy_score(y_sn2_17pt_test, nn_sn2_17pt_pred)
print("Test accuracy:", nn_acc_sn2_17pt)


In [ ]:
test_sn1_17pt = (X_sn1_17pt_test , y_sn1_17pt_test)

with open(os.path.join(data_arrays , "nn_test_sn1_17pt.pkl") , "wb") as f:
    pickle.dump(test_sn1_17pt, f)

test_sn2_17pt = (X_sn2_17pt_test , y_sn2_17pt_test)

with open(os.path.join(data_arrays , "nn_test_sn2_17pt.pkl") , "wb") as f:
    pickle.dump(test_sn2_17pt, f)

filename_nn_model_sn1_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn1_17pt_model_ft_last.sav')
filename_scaler_sn1_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn1_17pt_scaler_ft_last.sav')
pickle.dump(nn_sn1_17pt, open(filename_nn_model_sn1_17pt, 'wb'))
pickle.dump(scaler_sn1_17pt, open(filename_scaler_sn1_17pt, 'wb'))

filename_nn_model_sn2_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers",'nn_sn2_17pt_model_ft_last.sav')
filename_scaler_sn2_17pt = os.path.join(main_dir , "heavy_data_NO_GIT/models_scalers/",'nn_sn2_17pt_scaler_ft_last.sav')
pickle.dump(nn_sn2_17pt, open(filename_nn_model_sn2_17pt, 'wb'))
pickle.dump(scaler_sn2_17pt, open(filename_scaler_sn2_17pt, 'wb'))
